# Forecast Results and Penalty Occurrence Review

This notebook visualizes the saved forecast artifacts in `data/silver/forecast_artifacts` and compares actual delivery behavior against the planned delivery schedule generated by the optimizer.

## What this notebook covers

- Forecast model ranking by target series
- Direct-total versus summed-component forecast performance
- Actual usage, actual delivery, and planned delivery over the 2024 backtest window
- Month-end balance performance against Nicor inventory bands
- Daily and month-end tariff violation occurrence for actuals versus the planned schedule

## Violation logic

This notebook now tracks **whether a tariff violation occurred**, not an estimated cash-out amount.

- Daily activity violations are flagged when storage injection or withdrawal exceeds the month-specific Nicor limit
- Month-end inventory violations are flagged when the ending storage balance falls outside the Nicor band for that month
- Actual-delivery flags are taken directly from the reconstructed storage path so the benchmark uses the same boolean rule set as the package code
- Cumulative charts show the number of dates with at least one violation, not dollar severity

In [1]:
from pathlib import Path
import sys

import altair as alt
import pandas as pd
import polars as pl


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "silver" / "forecast_artifacts").exists():
            return candidate
    msg = "Could not locate repository root."
    raise FileNotFoundError(msg)


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

alt.data_transformers.disable_max_rows()

from dgup._internal.forecasting import _build_penalty_flag_report
from dgup._internal.storage import _reconstruct_storage
from dgup._internal.tariffs import _daily_storage_limits, _month_end_inventory_limits

In [11]:
ARTIFACT_DIR = REPO_ROOT / "data" / "silver" / "forecast_artifacts"
USAGE_PATH = REPO_ROOT / "data" / "silver" / "uta_gas_usage.parquet"

summary_frame = pl.read_parquet(ARTIFACT_DIR / "forecast-v2-summary.parquet").to_pandas()
aggregate_metrics = pl.read_parquet(ARTIFACT_DIR / "forecast-v2-aggregate.parquet").to_pandas()
delivery_plan = pl.read_parquet(ARTIFACT_DIR / "forecast-v2-delivery-plan.parquet").to_pandas()
delivery_summary = pl.read_parquet(ARTIFACT_DIR / "forecast-v2-delivery-summary.parquet").to_pandas()

usage_frame = (
    pl.read_parquet(USAGE_PATH)
    .with_columns(
        (pl.col("Usage - 1") + pl.col("Usage - 2") + pl.col("Usage - 2_1")).alias("Total Usage")
    )
    .to_pandas()
)

for frame in (summary_frame, aggregate_metrics, delivery_plan, delivery_summary, usage_frame):
    if "Date" in frame.columns:
        frame["Date"] = pd.to_datetime(frame["Date"])

window_start = delivery_plan["Date"].min()
window_end = delivery_plan["Date"].max()

actual_storage_full = _reconstruct_storage(
    pl.from_pandas(usage_frame[["Date", "Delivery", "Total Usage"]])
).to_pandas()
actual_storage_full["Date"] = pd.to_datetime(actual_storage_full["Date"])

FLAG_COLUMNS = [
    "injection_limit_exceeded",
    "withdrawal_limit_exceeded",
    "below_min_inventory",
    "above_max_inventory",
    "balance_below_zero",
    "balance_above_capacity",
]

actual_strategy = actual_storage_full.loc[
    actual_storage_full["Date"].between(window_start, window_end),
    [
        "Date",
        "Delivery",
        "Total Usage",
        "ending_balance",
        "max_injection",
        "max_withdrawal",
        "min_inventory",
        "max_inventory",
        "is_month_end",
        *FLAG_COLUMNS,
    ],
].copy()


def add_tariff_limits(frame: pd.DataFrame) -> pd.DataFrame:
    working = frame.copy()
    working["Date"] = pd.to_datetime(working["Date"])
    working["month"] = working["Date"].dt.month.astype(int)
    daily_limits = [_daily_storage_limits(month) for month in working["month"]]
    monthly_limits = [_month_end_inventory_limits(month) for month in working["month"]]
    working["max_injection"] = [limits[0] for limits in daily_limits]
    working["max_withdrawal"] = [limits[1] for limits in daily_limits]
    working["min_inventory"] = [limits[0] for limits in monthly_limits]
    working["max_inventory"] = [limits[1] for limits in monthly_limits]
    working["is_month_end"] = working["Date"].dt.is_month_end
    return working.sort_values("Date").reset_index(drop=True)


planned_balance_column = "ending_balance" if "ending_balance" in delivery_plan.columns else "realized_ending_balance"
planned_strategy = delivery_plan.loc[:, [
    "Date",
    "optimized_delivery",
    "actual_total_usage",
    planned_balance_column,
]].rename(
    columns={
        "optimized_delivery": "Delivery",
        "actual_total_usage": "Total Usage",
        planned_balance_column: "ending_balance",
    }
)
planned_strategy = add_tariff_limits(planned_strategy)
planned_strategy["storage_delta"] = planned_strategy["Delivery"] - planned_strategy["Total Usage"]
planned_strategy["injection"] = planned_strategy["storage_delta"].clip(lower=0.0)
planned_strategy["withdrawal"] = (-planned_strategy["storage_delta"]).clip(lower=0.0)
for column in FLAG_COLUMNS:
    if column in delivery_plan.columns:
        planned_strategy[column] = delivery_plan[column].astype(bool).to_numpy()

if "injection_limit_exceeded" not in planned_strategy.columns:
    planned_strategy["injection_limit_exceeded"] = planned_strategy["injection"] > planned_strategy["max_injection"]
if "withdrawal_limit_exceeded" not in planned_strategy.columns:
    planned_strategy["withdrawal_limit_exceeded"] = planned_strategy["withdrawal"] > planned_strategy["max_withdrawal"]
if "below_min_inventory" not in planned_strategy.columns:
    planned_strategy["below_min_inventory"] = planned_strategy["is_month_end"] & (planned_strategy["ending_balance"] < planned_strategy["min_inventory"])
if "above_max_inventory" not in planned_strategy.columns:
    planned_strategy["above_max_inventory"] = planned_strategy["is_month_end"] & (planned_strategy["ending_balance"] > planned_strategy["max_inventory"])
if "balance_below_zero" not in planned_strategy.columns:
    planned_strategy["balance_below_zero"] = planned_strategy["ending_balance"] < 0
if "balance_above_capacity" not in planned_strategy.columns:
    planned_strategy["balance_above_capacity"] = False

window_start, window_end, len(actual_strategy)

(Timestamp('2024-01-01 00:00:00'), Timestamp('2024-11-30 00:00:00'), 335)

In [12]:
daily_violations, monthly_violations, violation_totals, violation_by_date, month_end_balance = _build_penalty_flag_report(
    actual_strategy=actual_strategy,
    planned_strategy=planned_strategy,
    actual_label="Actual delivery",
    planned_label="Planned delivery",
)

window_delivery_summary = pd.DataFrame(
    [
        {
            "strategy": "actual_delivery",
            "mean_delivery": float(actual_strategy["Delivery"].mean()),
            "mean_ending_balance": float(actual_strategy["ending_balance"].mean()),
            "injection_limit_exceeded": int(actual_strategy["injection_limit_exceeded"].sum()),
            "withdrawal_limit_exceeded": int(actual_strategy["withdrawal_limit_exceeded"].sum()),
            "below_min_inventory": int(actual_strategy["below_min_inventory"].sum()),
            "above_max_inventory": int(actual_strategy["above_max_inventory"].sum()),
            "balance_below_zero": int(actual_strategy["balance_below_zero"].sum()),
            "balance_above_capacity": int(actual_strategy["balance_above_capacity"].sum()),
        },
        {
            "strategy": "optimized_delivery",
            "mean_delivery": float(planned_strategy["Delivery"].mean()),
            "mean_ending_balance": float(planned_strategy["ending_balance"].mean()),
            "injection_limit_exceeded": int(planned_strategy["injection_limit_exceeded"].sum()),
            "withdrawal_limit_exceeded": int(planned_strategy["withdrawal_limit_exceeded"].sum()),
            "below_min_inventory": int(planned_strategy["below_min_inventory"].sum()),
            "above_max_inventory": int(planned_strategy["above_max_inventory"].sum()),
            "balance_below_zero": int(planned_strategy["balance_below_zero"].sum()),
            "balance_above_capacity": int(planned_strategy["balance_above_capacity"].sum()),
        },
    ]
)

delivery_compare = pd.concat(
    [
        actual_strategy.loc[:, ["Date", "Delivery"]].rename(columns={"Delivery": "therms"}).assign(series="Actual delivery"),
        planned_strategy.loc[:, ["Date", "Delivery"]].rename(columns={"Delivery": "therms"}).assign(series="Planned delivery"),
        actual_strategy.loc[:, ["Date", "Total Usage"]].rename(columns={"Total Usage": "therms"}).assign(series="Actual usage"),
    ],
    ignore_index=True,
)

violation_totals

,strategy,violation_type,violation_count
0,Actual delivery,Daily activity violation,136
1,Planned delivery,Daily activity violation,93
2,Actual delivery,Month-end inventory violation,2
3,Planned delivery,Month-end inventory violation,10
4,Actual delivery,Any penalty day,137
5,Planned delivery,Any penalty day,102


In [13]:
best_models = (
    summary_frame.sort_values(["series", "rank"])
    .groupby("series", group_keys=False)
    .head(3)
    .loc[:, ["series", "rank", "model", "mae", "smape", "rmse", "bias"]]
    .reset_index(drop=True)
)

violation_summary_table = (
    violation_totals.pivot(index="strategy", columns="violation_type", values="violation_count")
    .fillna(0)
    .astype(int)
)

display(best_models.round(3))
display(aggregate_metrics.round(3))
display(window_delivery_summary.round(3))
violation_summary_table

,series,rank,model,mae,smape,rmse,bias
0,Total Usage,1.0,lightgbm,186.966,12.721,254.620,-9.053
1,Total Usage,2.0,xgboost,190.159,12.975,255.536,-34.075
2,Total Usage,3.0,hist_gbm,191.790,12.959,259.648,-11.406
3,Usage - 1,1.0,hist_gbm,31.485,20.810,40.921,5.896
4,Usage - 1,2.0,lightgbm,31.572,20.908,41.268,5.223
5,Usage - 1,3.0,xgboost,32.484,22.486,41.550,5.875
6,Usage - 2,1.0,hist_gbm,177.859,13.756,240.747,-14.159
7,Usage - 2,2.0,lightgbm,179.960,13.816,240.350,-12.353
8,Usage - 2,3.0,xgboost,182.688,14.346,240.601,-38.738
9,Usage - 2_1,1.0,mlp,3.013,122.072,5.125,-0.886


,strategy,mae,rmse,smape,mape,bias
0,sum_of_component_bests,186.948,255.052,12.712,16.658,-9.149
1,direct_total_best,186.966,254.620,12.721,17.669,-9.053


,strategy,mean_delivery,mean_ending_balance,injection_limit_exceeded,withdrawal_limit_exceeded,below_min_inventory,above_max_inventory,balance_below_zero,balance_above_capacity
0,actual_delivery,1870.955,61853.786,88,48,0,2,0,0
1,optimized_delivery,1815.179,40465.879,55,38,10,0,72,0


violation_type,Any penalty day,Daily activity violation,Month-end inventory violation
strategy,,,
Actual delivery,137,136,2
Planned delivery,102,93,10


In [5]:
model_chart = (
    alt.Chart(summary_frame)
    .mark_bar()
    .encode(
        x=alt.X("mae:Q", title="MAE"),
        y=alt.Y("model:N", sort="-x", title="Model"),
        color=alt.Color("model:N", legend=None),
        tooltip=["series:N", "model:N", alt.Tooltip("mae:Q", format=".2f"), alt.Tooltip("smape:Q", format=".2f")],
    )
    .properties(width=210, height=180, title="Model ranking by target series")
    .facet(column=alt.Column("series:N", title=None))
)

aggregate_chart = (
    alt.Chart(aggregate_metrics)
    .mark_bar(size=60)
    .encode(
        x=alt.X("strategy:N", title=None),
        y=alt.Y("mae:Q", title="MAE"),
        color=alt.Color("strategy:N", title="Forecast strategy"),
        tooltip=["strategy:N", alt.Tooltip("mae:Q", format=".2f"), alt.Tooltip("smape:Q", format=".2f"), alt.Tooltip("bias:Q", format=".2f")],
    )
    .properties(width=320, height=240, title="Direct-total vs component-sum performance")
)

model_chart & aggregate_chart

alt.VConcatChart(...)

In [14]:
delivery_chart = (
    alt.Chart(delivery_compare)
    .mark_line()
    .encode(
        x=alt.X("Date:T", title=None),
        y=alt.Y("therms:Q", title="Therms"),
        color=alt.Color("series:N", title="Series"),
        tooltip=["Date:T", "series:N", alt.Tooltip("therms:Q", format=".1f")],
    )
    .properties(width=900, height=320, title="Actual usage vs actual and planned delivery")
)

band_frame = month_end_balance.loc[:, ["Date", "min_inventory", "max_inventory"]].drop_duplicates()
band_lower = alt.Chart(band_frame).mark_line(strokeDash=[4, 4], color="#7f7f7f").encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("min_inventory:Q", title="Ending balance"),
    tooltip=["Date:T", alt.Tooltip("min_inventory:Q", format=".1f")],
)
band_upper = alt.Chart(band_frame).mark_line(strokeDash=[4, 4], color="#7f7f7f").encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("max_inventory:Q", title="Ending balance"),
    tooltip=["Date:T", alt.Tooltip("max_inventory:Q", format=".1f")],
)
balance_lines = alt.Chart(month_end_balance).mark_line(point=True).encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("ending_balance:Q", title="Ending balance"),
    color=alt.Color("strategy:N", title="Strategy"),
    tooltip=["Date:T", "strategy:N", alt.Tooltip("ending_balance:Q", format=".1f")],
)

month_end_chart = (band_lower + band_upper + balance_lines).properties(
    width=900,
    height=260,
    title="Month-end balance against Nicor inventory bands",
)

delivery_chart & month_end_chart

alt.VConcatChart(...)

In [15]:
violation_totals_chart = (
    alt.Chart(violation_totals.loc[violation_totals["violation_type"] != "Any penalty day"] )
    .mark_bar(size=42)
    .encode(
        x=alt.X("strategy:N", title=None),
        y=alt.Y("violation_count:Q", title="Violation count"),
        color=alt.Color("violation_type:N", title="Violation type"),
        xOffset="violation_type:N",
        tooltip=["strategy:N", "violation_type:N", "violation_count:Q"],
    )
    .properties(width=360, height=260, title="Actual vs planned violation counts")
)

cumulative_violation_chart = (
    alt.Chart(violation_by_date)
    .mark_line()
    .encode(
        x=alt.X("Date:T", title=None),
        y=alt.Y("cumulative_penalty_days:Q", title="Cumulative violation days"),
        color=alt.Color("strategy:N", title="Strategy"),
        tooltip=["Date:T", "strategy:N", "cumulative_penalty_days:Q", "penalty_occurred:N"],
    )
    .properties(width=520, height=260, title="Cumulative dates with any tariff violation")
)

month_end_violation_detail = pd.concat(
    [
        monthly_violations.loc[:, ["Date", "strategy", "monthly_violation_occurred", "monthly_violation_class"]],
        monthly_violations.loc[:, ["Date", "strategy", "monthly_violation_occurred", "monthly_violation_class"]],
    ],
    ignore_index=True,
).drop_duplicates()
month_end_violation_detail["violation_flag"] = month_end_violation_detail["monthly_violation_occurred"].astype(int)

month_end_violation_chart = (
    alt.Chart(month_end_violation_detail)
    .mark_bar(size=18)
    .encode(
        x=alt.X("Date:T", title=None),
        y=alt.Y("violation_flag:Q", title="Month-end violation occurred"),
        color=alt.Color("strategy:N", title="Strategy"),
        xOffset="strategy:N",
        tooltip=["Date:T", "strategy:N", "monthly_violation_class:N", "violation_flag:Q"],
    )
    .properties(width=900, height=220, title="Month-end violation detail (1 = violation)")
)

(violation_totals_chart | cumulative_violation_chart) & month_end_violation_chart

alt.VConcatChart(...)

In [16]:
flagged_daily_violations = daily_violations.loc[
    daily_violations["daily_violation_occurred"],
    [
        "Date",
        "strategy",
        "daily_violation_class",
        "injection_limit_exceeded",
        "withdrawal_limit_exceeded",
        "excess_injection",
        "excess_withdrawal",
    ],
]
flagged_daily_violations = flagged_daily_violations.sort_values(["strategy", "Date"]).reset_index(drop=True)

month_end_detail = monthly_violations.loc[:, [
    "Date",
    "strategy",
    "ending_balance",
    "min_inventory",
    "max_inventory",
    "monthly_violation_occurred",
    "monthly_violation_class",
]]
month_end_detail = month_end_detail.sort_values(["strategy", "Date"]).reset_index(drop=True)

flagged_daily_violations_display = flagged_daily_violations.copy()
for column in ["excess_injection", "excess_withdrawal"]:
    flagged_daily_violations_display[column] = flagged_daily_violations_display[column].round(2)

month_end_detail_display = month_end_detail.copy()
for column in ["ending_balance", "min_inventory", "max_inventory"]:
    month_end_detail_display[column] = month_end_detail_display[column].round(2)

display(flagged_daily_violations_display.head(25))
month_end_detail_display

,Date,strategy,daily_violation_class,injection_limit_exceeded,withdrawal_limit_exceeded,excess_injection,excess_withdrawal
0,2024-01-02,Actual delivery,Over-withdrawal,False,True,0.0,384.05
1,2024-01-03,Actual delivery,Over-withdrawal,False,True,0.0,101.51
2,2024-01-04,Actual delivery,Over-withdrawal,False,True,0.0,320.97
3,2024-01-09,Actual delivery,Over-withdrawal,False,True,0.0,170.48
4,2024-01-10,Actual delivery,Over-withdrawal,False,True,0.0,249.89
5,2024-01-15,Actual delivery,Over-withdrawal,False,True,0.0,376.91
6,2024-01-16,Actual delivery,Over-withdrawal,False,True,0.0,280.76
7,2024-01-18,Actual delivery,Over-withdrawal,False,True,0.0,101.52
8,2024-01-22,Actual delivery,Over-withdrawal,False,True,0.0,12.02
9,2024-01-29,Actual delivery,Over-withdrawal,False,True,0.0,328.66


,Date,strategy,ending_balance,min_inventory,max_inventory,monthly_violation_occurred,monthly_violation_class
0,2024-01-31,Actual delivery,59442.81,50694.35,65178.45,False,Within band
1,2024-02-29,Actual delivery,20950.32,14484.10,36210.25,False,Within band
2,2024-03-31,Actual delivery,10856.54,0.00,14484.10,False,Within band
3,2024-04-30,Actual delivery,8960.68,0.00,14484.10,False,Within band
4,2024-05-31,Actual delivery,26647.94,14484.10,28968.20,False,Within band
5,2024-06-30,Actual delivery,38483.78,28968.20,43452.30,False,Within band
6,2024-07-31,Actual delivery,55949.92,43452.30,57936.40,False,Within band
7,2024-08-31,Actual delivery,85768.74,72420.50,86904.60,False,Within band
8,2024-09-30,Actual delivery,116359.82,101388.70,115872.80,True,Above maximum
9,2024-10-31,Actual delivery,144253.63,123114.85,144841.00,False,Within band


In [17]:
from IPython.display import Markdown, display

actual_daily_violations = int(violation_summary_table.loc["Actual delivery", "Daily activity violation"])
planned_daily_violations = int(violation_summary_table.loc["Planned delivery", "Daily activity violation"])
actual_month_end_violations = int(violation_summary_table.loc["Actual delivery", "Month-end inventory violation"])
planned_month_end_violations = int(violation_summary_table.loc["Planned delivery", "Month-end inventory violation"])
actual_penalty_days = int(violation_summary_table.loc["Actual delivery", "Any penalty day"])
planned_penalty_days = int(violation_summary_table.loc["Planned delivery", "Any penalty day"])

optimized_row = window_delivery_summary.loc[window_delivery_summary["strategy"] == "optimized_delivery"].iloc[0]
penalty_day_change = actual_penalty_days - planned_penalty_days
negative_balance_days = int(optimized_row["balance_below_zero"])

presentation = f"""
## Business Takeaways and Recommendations

- Using boolean tariff flags instead of synthetic cash-out amounts, actual delivery records **{actual_daily_violations}** daily activity violations and **{actual_month_end_violations}** month-end inventory violations in the 2024 backtest window.
- The optimized delivery plan records **{planned_daily_violations}** daily activity violations and **{planned_month_end_violations}** month-end inventory violations.
- Counting unique dates with any tariff issue, the optimized plan changes the total from **{actual_penalty_days}** days under actual delivery to **{planned_penalty_days}** days under the planned schedule, a difference of **{penalty_day_change}** days.
- The remaining operational weakness is storage-bank depletion risk. The optimized plan still records **{negative_balance_days}** days with `balance_below_zero`, even when tariff-violation days improve.

### What Changed in the Review

- The notebook now treats penalty analysis as a flag problem: either a tariff violation happened or it did not.
- The actual-delivery flags now come directly from the reconstructed storage path instead of inferred cash-out severities.
- The benchmark summary table now uses the same 2024 backtest window as the violation counts, so the totals are internally consistent.

### Recommendations

1. Keep the existing optimizer as the benchmark policy for reducing violation days, not estimated cash-out dollars.
2. Use the separate rolling LightGBM prototype notebook to test a more conservative operating policy with weekly refits, short-horizon forecasting, reserve floors, and fallback demand assumptions.
3. Treat `balance_below_zero` as a hard operational warning even if the violation-day count improves.
4. If you later need cost estimates, layer price data on top of these corrected boolean flags rather than using synthetic cash-out formulas in the notebook.
"""

display(Markdown(presentation))


## Business Takeaways and Recommendations

- Using boolean tariff flags instead of synthetic cash-out amounts, actual delivery records **136** daily activity violations and **2** month-end inventory violations in the 2024 backtest window.
- The optimized delivery plan records **93** daily activity violations and **10** month-end inventory violations.
- Counting unique dates with any tariff issue, the optimized plan changes the total from **137** days under actual delivery to **102** days under the planned schedule, a difference of **35** days.
- The remaining operational weakness is storage-bank depletion risk. The optimized plan still records **72** days with `balance_below_zero`, even when tariff-violation days improve.

### What Changed in the Review

- The notebook now treats penalty analysis as a flag problem: either a tariff violation happened or it did not.
- The actual-delivery flags now come directly from the reconstructed storage path instead of inferred cash-out severities.
- The benchmark summary table now uses the same 2024 backtest window as the violation counts, so the totals are internally consistent.

### Recommendations

1. Keep the existing optimizer as the benchmark policy for reducing violation days, not estimated cash-out dollars.
2. Use the separate rolling LightGBM prototype notebook to test a more conservative operating policy with weekly refits, short-horizon forecasting, reserve floors, and fallback demand assumptions.
3. Treat `balance_below_zero` as a hard operational warning even if the violation-day count improves.
4. If you later need cost estimates, layer price data on top of these corrected boolean flags rather than using synthetic cash-out formulas in the notebook.
